In [ ]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

# Find the repo root from wherever this notebook runs (fixes the relative-path bug from M0)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "meridian.duckdb"), read_only=True)

def q(sql: str) -> pl.DataFrame:
    return con.sql(sql).pl()

print(q("select count(*) as sales, min(sale_date) as first_day, max(sale_date) as last_day from main_marts.fct_sales"))

In [ ]:
kpi = q("""
    with orders as (                       -- an 'order' = one customer on one day
        select customer_id, sale_date,
               date_trunc('month', sale_date)::date as month,
               count(*) as units, sum(price_index) as revenue
        from main_marts.fct_sales
        group by all
    )
    select month,
           count(distinct customer_id)              as customers,
           count(*)                                 as orders,
           sum(units)                               as units,
           sum(revenue)                             as revenue,
           count(*) / count(distinct customer_id)   as frequency,     -- orders per customer
           sum(revenue) / count(*)                  as basket_value   -- revenue per order
    from orders
    where month between date '2018-10-01' and date '2020-08-01'   -- complete months only
    group by month
    order by month
""")
check = (kpi["customers"] * kpi["frequency"] * kpi["basket_value"] - kpi["revenue"]).abs().max()
print(f"identity check (should be ~0): {check:.10f}")
kpi.head()

In [ ]:
idx = kpi.select(
    "month",
    *[(pl.col(c) / pl.col(c).first() * 100).alias(c) for c in ["revenue", "customers", "frequency", "basket_value"]],
)
fig = px.line(
    idx.unpivot(index="month", variable_name="kpi", value_name="index"),
    x="month", y="index", color="kpi",
    title="Revenue and its 3 drivers, indexed (Oct 2018 = 100)",
)
fig.add_hline(y=100, line_dash="dot", line_color="grey")
fig.show()

In [ ]:
yoy = (
    kpi.with_columns(pl.col("month").dt.offset_by("-1y").alias("month_ly"))
    .join(kpi, left_on="month_ly", right_on="month", suffix="_ly")
    .select(
        "month",
        ((pl.col("revenue") / pl.col("revenue_ly")).log() * 100).alias("revenue_pct_log"),
        ((pl.col("customers") / pl.col("customers_ly")).log() * 100).alias("customers"),
        ((pl.col("frequency") / pl.col("frequency_ly")).log() * 100).alias("frequency"),
        ((pl.col("basket_value") / pl.col("basket_value_ly")).log() * 100).alias("basket_value"),
    )
)
fig = go.Figure()
for c in ["customers", "frequency", "basket_value"]:
    fig.add_bar(x=yoy["month"], y=yoy[c], name=c)
fig.add_scatter(x=yoy["month"], y=yoy["revenue_pct_log"], name="total revenue change",
                mode="lines+markers", line=dict(color="black"))
fig.update_layout(barmode="relative", title="YoY revenue change split by driver (log %, bars sum to the line)",
                  yaxis_title="YoY change, log %")
fig.show()
yoy

In [ ]:
ret = q("""
    with active as (
        select distinct customer_id, date_trunc('month', sale_date)::date as month
        from main_marts.fct_sales
    ),
    first_seen as (
        select customer_id, min(month) as cohort from active group by 1
    )
    select f.cohort,
           datediff('month', f.cohort, a.month)     as months_since,
           count(*)                                 as active_customers
    from active a join first_seen f using (customer_id)
    group by all
""")
size = ret.filter(pl.col("months_since") == 0).select("cohort", pl.col("active_customers").alias("cohort_size"))
grid = (
    ret.join(size, on="cohort")
    .with_columns((pl.col("active_customers") / pl.col("cohort_size") * 100).round(1).alias("retention_pct"))
    .filter((pl.col("months_since").is_between(1, 12)) & (pl.col("cohort") >= pl.date(2019, 1, 1)))
    .pivot(on="months_since", index="cohort", values="retention_pct", sort_columns=True)
    .sort("cohort")
)
fig = px.imshow(
    grid.drop("cohort").to_numpy(),
    x=[str(c) for c in grid.columns[1:]], y=[str(d) for d in grid["cohort"]],
    labels=dict(x="months since first purchase", y="cohort (first month seen)", color="% active"),
    color_continuous_scale="Blues", text_auto=True, aspect="auto",
    title="Monthly cohort retention, % of cohort buying again",
)
fig.show()

In [ ]:
grid.select("cohort", pl.col("1").alias("m1_pct"), pl.col("3").alias("m3_pct"))

In [ ]:
pl.Config.set_tbl_rows(30)
print(kpi.select("month", "customers", pl.col("frequency").round(3), pl.col("basket_value").round(4)))
print(yoy.select(pl.all().exclude("month").round(1), "month"))
print(grid.select("cohort", "1", "3", "6"))

In [ ]:
pl.Config.set_tbl_rows(30)

basket = (
    kpi.with_columns(
        (pl.col("units") / pl.col("orders")).alias("units_per_order"),
        (pl.col("revenue") / pl.col("units")).alias("price_per_unit"),
    )
)
b_yoy = (
    basket.with_columns(pl.col("month").dt.offset_by("-1y").alias("month_ly"))
    .join(basket, left_on="month_ly", right_on="month", suffix="_ly")
    .select(
        "month",
        ((pl.col("basket_value") / pl.col("basket_value_ly")).log() * 100).round(1).alias("basket_value"),
        ((pl.col("units_per_order") / pl.col("units_per_order_ly")).log() * 100).round(1).alias("units_per_order"),
        ((pl.col("price_per_unit") / pl.col("price_per_unit_ly")).log() * 100).round(1).alias("price_per_unit"),
    )
)
fig = go.Figure()
for c in ["units_per_order", "price_per_unit"]:
    fig.add_bar(x=b_yoy["month"], y=b_yoy[c], name=c)
fig.add_scatter(x=b_yoy["month"], y=b_yoy["basket_value"], name="basket value change",
                mode="lines+markers", line=dict(color="black"))
fig.update_layout(barmode="relative", title="YoY basket change = more items per order + higher price per item (log %)",
                  yaxis_title="YoY change, log %")
fig.show()
print(b_yoy)

In [ ]:
ch = q("""
    with orders as (
        select customer_id, sale_date, sales_channel,
               date_trunc('month', sale_date)::date as month,
               count(*) as units, sum(price_index) as revenue
        from main_marts.fct_sales
        group by all
    )
    select month, sales_channel,
           count(*)                 as orders,
           sum(revenue)             as revenue,
           sum(revenue) / count(*)  as basket_value,
           sum(units) / count(*)    as units_per_order
    from orders
    where month between date '2018-10-01' and date '2020-08-01'
    group by all
    order by month, sales_channel
""")
share = (
    ch.group_by("month").agg(
        (pl.col("revenue").filter(pl.col("sales_channel") == "online").sum() / pl.col("revenue").sum() * 100)
        .round(1).alias("online_revenue_share_pct")
    ).sort("month")
)
fig = px.line(ch, x="month", y="revenue", color="sales_channel", markers=True,
              title="Monthly revenue by channel (price index units)")
fig.show()
fig = px.line(share, x="month", y="online_revenue_share_pct", markers=True,
              title="Online share of revenue, %")
fig.show()
print(share)

In [ ]:
print(ch.filter(pl.col("month").is_between(pl.date(2020, 1, 1), pl.date(2020, 6, 1)))
        .select("month", "sales_channel", "orders", pl.col("basket_value").round(4), pl.col("units_per_order").round(2)))

In [ ]:
from statsmodels.tsa.seasonal import STL

weekly = q("""
    select week_start, sum(price_index) as revenue
    from main_marts.fct_sales
    where week_start between date '2018-09-24' and date '2020-09-14'   -- full weeks only
    group by 1 order by 1
""")
print(f"{weekly.height} full weeks")
stl = STL(weekly["revenue"].to_numpy(), period=52, robust=True).fit()
parts = weekly.with_columns(
    pl.Series("trend", stl.trend),
    pl.Series("seasonal", stl.seasonal),
    pl.Series("leftover", stl.resid),
)
fig = px.line(parts.unpivot(index="week_start", on=["revenue", "trend", "seasonal", "leftover"]),
              x="week_start", y="value", facet_row="variable", height=800,
              title="Weekly revenue = trend + yearly seasonal pattern + leftover")
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
print(f"trend change, first to last full week: {(trend[-1] / trend[0] - 1) * 100:.1f}%")
print(f"seasonal swing, peak to trough: {seasonal_pct.max() - seasonal_pct.min():.0f} percentage points")
wk = weekly["week_start"]
print(f"peak season week: {wk[int(np.argmax(seasonal_pct[:52]))]:%d %b}, "
      f"low season week: {wk[int(np.argmin(seasonal_pct[:52]))]:%d %b}")
print("biggest leftover shocks (weeks the smooth pattern can't explain):")
print(parts.sort(pl.col("leftover_pct").abs(), descending=True)
           .select("week_start", pl.col("leftover_pct").round(1)).head(8))

In [ ]:
import numpy as np
import statsmodels.api as sm

weekly = q("""
    select week_start, sum(price_index) as revenue
    from main_marts.fct_sales
    where week_start between date '2018-09-24' and date '2020-09-14'   -- full weeks only
    group by 1 order by 1
""")
y = np.log(weekly["revenue"].to_numpy())               # log: effects become % changes
n = len(y)
tt = np.arange(n) / n                                   # time scaled 0..1
K = 3                                                   # 3 yearly waves = smooth season
fourier = np.column_stack([f(2 * np.pi * k * np.arange(n) / 52) for k in range(1, K + 1) for f in (np.sin, np.cos)])
trend_X = np.column_stack([tt, tt**2, tt**3])           # smooth curved trend
X = sm.add_constant(np.column_stack([trend_X, fourier]))
fit = sm.RLM(y, X, M=sm.robust.norms.HuberT()).fit()    # robust: shocks can't pull the fit
b = fit.params
trend = np.exp(b[0] + trend_X @ b[1:4])
seasonal_pct = (np.exp(fourier @ b[4:]) - 1) * 100
leftover_pct = (np.exp(y - X @ b) - 1) * 100
parts = weekly.with_columns(
    pl.Series("trend", trend),
    pl.Series("seasonal_pct", seasonal_pct),
    pl.Series("leftover_pct", leftover_pct),
)
fig = px.line(parts.unpivot(index="week_start", on=["revenue", "trend", "seasonal_pct", "leftover_pct"]),
              x="week_start", y="value", facet_row="variable", height=800,
              title="Weekly revenue = smooth trend x smooth yearly season x leftover shocks")
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
ws = weekly["week_start"].to_numpy()
black_friday = np.isin(ws, np.array(["2018-11-19", "2019-11-25"], dtype="datetime64[D]")).astype(float)
lockdown = ((ws >= np.datetime64("2020-03-16")) & (ws <= np.datetime64("2020-05-04"))).astype(float)
X = sm.add_constant(np.column_stack([trend_X, fourier, black_friday, lockdown]))

In [ ]:
print(f"Black Friday week effect: {(np.exp(b[-2]) - 1) * 100:+.0f}%")
print(f"Lockdown weeks effect:    {(np.exp(b[-1]) - 1) * 100:+.0f}%")

In [ ]:
print(len(b), X.shape)

In [ ]:
trend_X = np.column_stack([tt, tt**2, tt**3])           # smooth curved trend
ws = weekly["week_start"].to_numpy()
black_friday = np.isin(ws, np.array(["2018-11-19", "2019-11-25"], dtype="datetime64[D]")).astype(float)
lockdown = ((ws >= np.datetime64("2020-03-16")) & (ws <= np.datetime64("2020-05-04"))).astype(float)
X = sm.add_constant(np.column_stack([trend_X, fourier, black_friday, lockdown]))
fit = sm.RLM(y, X, M=sm.robust.norms.HuberT()).fit()

In [ ]:
import numpy as np
import statsmodels.api as sm

weekly = q("""
    select week_start, sum(price_index) as revenue
    from main_marts.fct_sales
    where week_start between date '2018-09-24' and date '2020-09-14'   -- full weeks only
    group by 1 order by 1
""")
y = np.log(weekly["revenue"].to_numpy())
n = len(y)
tt = np.arange(n) / n
K = 3
fourier = np.column_stack([f(2 * np.pi * k * np.arange(n) / 52) for k in range(1, K + 1) for f in (np.sin, np.cos)])
trend_X = np.column_stack([tt, tt**2, tt**3])

ws = weekly["week_start"].to_numpy()
black_friday = np.isin(ws, np.array(["2018-11-19", "2019-11-25"], dtype="datetime64[D]")).astype(float)
lockdown = ((ws >= np.datetime64("2020-03-16")) & (ws <= np.datetime64("2020-05-04"))).astype(float)

# column order: [const | 3 trend | 6 fourier | black_friday | lockdown] = 12
X = sm.add_constant(np.column_stack([trend_X, fourier, black_friday, lockdown]))
fit = sm.RLM(y, X, M=sm.robust.norms.HuberT()).fit()
b = fit.params
assert len(b) == X.shape[1] == 12, "model and design matrix out of sync"

b_const, b_trend, b_four, b_bf, b_lock = b[0], b[1:4], b[4:10], b[10], b[11]
trend = np.exp(b_const + trend_X @ b_trend)
seasonal_pct = (np.exp(fourier @ b_four) - 1) * 100
leftover_pct = (np.exp(y - X @ b) - 1) * 100
parts = weekly.with_columns(
    pl.Series("trend", trend),
    pl.Series("seasonal_pct", seasonal_pct),
    pl.Series("leftover_pct", leftover_pct),
)
fig = px.line(parts.unpivot(index="week_start", on=["revenue", "trend", "seasonal_pct", "leftover_pct"]),
              x="week_start", y="value", facet_row="variable", height=800,
              title="Weekly revenue = trend x season x events x leftover")
fig.update_yaxes(matches=None)
fig.show()